# 15 · Is it the shape of my approach, or how it is executed?

Notebooks 12-14: copying my approach to the bowl gives 66% (two seeds), planning it straight down gives 98%. The two approaches differ in several ways at once. This run keeps the **shape** of my approach (the same path, sideways and just above the rim) and changes **how it is walked**: resampled to 8 evenly spaced points and advanced on a fixed clock (4 steps per point), instead of point-by-point "go until you get there".

What that changes, from physics-only checks of the teacher in the evaluation scenes (5 demos x 25 scenes):

| | my approach | **fixed-pace approach (this run)** | planned straight down |
|---|---|---|---|
| teacher success | 98% | 98% | 98% |
| where it closes (from the bowl's axis) | 5.1 cm | 4.4 cm | 4.4 cm |
| when it closes, median per demo (steps) | 54-72 | 63-69 | 45 |
| spread within the erratic demo `f911` (steps) | 60 | 9 | 1 |

So this run keeps the sideways path but fixes the execution: a regular moment of closing and the same grasp margin as the planned approach. It cannot separate timing from margin; it separates **path shape** from **execution**.

**Predictions, fixed before running** (pooled over 2 evaluations, n = 100; same scenes, seeds and dataset size as notebooks 09 and 12):
* >= 90%: the sideways shape is learnable; what hurt was how my approach was executed (irregular timing and/or closing near the rim's edge). Human approach paths can be kept if they are executed consistently.
* <= 75%: the shape itself (coming in sideways, just above the rim) is what a policy cannot learn here.
* In between: both contribute.

**Runtime:** T4, about 3 h.

In [ ]:
import os, sys, json, subprocess
os.environ["MUJOCO_GL"] = "egl"                      # GPU rendering, must be set before mujoco is imported
REPO_URL = "https://github.com/LukasWill/phone2panda.git"
BASE = "/content" if os.path.exists("/content") else "/kaggle/working"   # Colab or Kaggle
CODE_DIR = f"{BASE}/phone2panda"
if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "-q"], check=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lerobot[libero]==0.6.1", "imageio-ffmpeg"],
                   capture_output=True, text=True)
if r.returncode != 0:                                  # show pip's own reason, not just "exit status 1"
    print((r.stdout + r.stderr)[-4000:]); raise RuntimeError("pip install failed: see the lines above")
# Colab preinstalls an older torchao; a newer diffusers (pulled in by lerobot) then fails on import
# (FqnToConfig). ACT does not use torchao, and diffusers skips it when it is absent.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
from p2p.sim.env import ensure_nvidia_egl
ensure_nvidia_egl()      # Colab: make EGL render on the GPU (otherwise it silently renders on the CPU, ~10x slower)
# Hugging Face token (Colab: key icon -> Secrets -> HF_TOKEN; Kaggle: Add-ons -> Secrets -> HF_TOKEN)
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN found: uploads to the Hub will fail")
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import whoami
    print("Hugging Face account:", whoami(token=os.environ["HF_TOKEN"])["name"])
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
VARIANT = "timed"
STRATEGY = "object_relative_timed"
DEMOS = "f9115215f9_lefthand,c5f525e833,9a63cb1538_lefthand,92e0b86bc6_lefthand,65ee6ec786"   # grasp sides 106-116°
PER_DEMO, N_EPISODES, SEED0 = 130, 450, 2_000_000     # the same seeds as notebook 09: same scenes, same perturbations
HUB_REPO, SEED, STEPS, BATCH = "phone2panda-data", 0, 20000, 32
NAME = f"ours_consistent_{VARIANT}"
RUN = f"act_{NAME}_s{SEED}"
GEN = f"{BASE}/gen_{VARIANT}"
WORKERS = os.cpu_count()
from p2p.sim.env import render_speed
r = render_speed(); print(r)
assert r["gpu_rendering"], "rendering is on the CPU: Runtime -> Change runtime type -> T4 GPU, then restart"

## Generate (resumable: rerun the cell after a disconnect and it continues)

In [ ]:
!python -m p2p.data.generate data/processed {GEN} --strategy {STRATEGY} --demos {DEMOS} --per-demo {PER_DEMO} --seed0 {SEED0} --workers {WORKERS}

In [ ]:
import pandas as pd
log = pd.read_json(f"{GEN}/generation_log.jsonl", lines=True).drop_duplicates("seed", keep="last")
print(VARIANT, "yield", round(log.success.mean(), 3), "successes", int(log.success.sum()), "of", len(log))
display(log.groupby("demo").success.agg(["mean", "sum"]))
assert log.success.sum() >= N_EPISODES, "fewer successes than N_EPISODES: lower N_EPISODES in BOTH variants"

## Dataset: 450 successes spread evenly over the demos, as in notebook 09

In [ ]:
!python -m p2p.data.lerobot_io ours {GEN} {BASE}/lerobot/{NAME} --balanced {N_EPISODES}
!python tools/demo_consistency.py {BASE}/lerobot/{NAME}
from p2p.hub import push
push(f"{BASE}/lerobot/{NAME}", HUB_REPO, path_in_repo=f"lerobot/{NAME}")

## Train and evaluate (same settings as every other run; evaluation now also records behaviour)

In [ ]:
!python -m p2p.policy.act train {BASE}/lerobot/{NAME} {BASE}/runs/{RUN} --steps {STEPS} --batch-size {BATCH} --seed {SEED}

In [ ]:
!python -m p2p.policy.act eval {BASE}/runs/{RUN}/final --episodes 50 --videos 6 --out {BASE}/runs/{RUN}/eval
!python -m p2p.policy.act eval {BASE}/runs/{RUN}/final --episodes 50 --videos 0 --out {BASE}/runs/{RUN}/eval_rerun
!python -m p2p.policy.act eval {BASE}/runs/{RUN}/final --episodes 50 --videos 0 --temporal-ensemble 0.01 --out {BASE}/runs/{RUN}/eval_te
import glob
evals = " ".join(sorted(glob.glob(f"{BASE}/runs/{RUN}/eval*/eval.json")))
!python tools/policy_failures.py {evals}
for e in sorted(glob.glob(f"{BASE}/runs/{RUN}/eval*/eval.json")):
    print(e.split("/")[-2], json.dumps(json.load(open(e))["behaviour"]))

In [ ]:
push(f"{BASE}/runs/{RUN}", HUB_REPO, path_in_repo=f"runs/{RUN}")
!cp {GEN}/generation_log.jsonl {BASE}/runs/{RUN}/generation_log.jsonl
!cd {BASE}/runs && zip -qr {BASE}/{RUN}.zip {RUN} -x "*/final/*" "*/step_*"
try:
    from google.colab import files; files.download(f"{BASE}/{RUN}.zip")
except Exception:
    print("download", f"{BASE}/{RUN}.zip", "from the Output panel")